### =============================================================================
### 1. IMPORTATIONS ET CONFIGURATION
### =============================================================================

In [ ]:
!pip install -q pandas transformers scikit-learn seaborn tqdm torch

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Vérification de la disponibilité du GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Dispositif utilisé: {device}")
if torch.cuda.is_available():
    print(f"   GPU détecté: {torch.cuda.get_device_name(0)}")
    print(f"   Mémoire disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("   ⚠️  Aucun GPU détecté, utilisation du CPU")

### =============================================================================
### 2. CHARGEMENT ET EXPLORATION DES DONNÉES
### =============================================================================

In [ ]:
print("\n" + "="*80)
print("CHARGEMENT DES DONNÉES")
print("="*80)

# Charger le dataset d'interactions (reviews)
# Note: Ajustez le chemin selon votre environnement
# Sur Kaggle: /kaggle/input/food-com-recipes-and-user-interactions/RAW_interactions.csv
df_reviews = pd.read_csv('/kaggle/input/mangetamain-dataset/RAW_interactions.csv')

print(f"\n📊 Shape du dataset: {df_reviews.shape}")
print(f"   Nombre de reviews: {len(df_reviews):,}")
print(f"\n📋 Colonnes disponibles:")
print(df_reviews.columns.tolist())

# Afficher les premières lignes
print(f"\n👀 Aperçu des données:")
print(df_reviews.head())

# Statistiques descriptives
print(f"\n📈 Distribution des notes (rating):")
print(df_reviews['rating'].value_counts().sort_index())

# Vérifier les valeurs manquantes
print(f"\n🔍 Valeurs manquantes:")
print(df_reviews.isnull().sum())

### =============================================================================
### 3. PRÉTRAITEMENT DES DONNÉES
### =============================================================================

In [ ]:
print("\n" + "="*80)
print("PRÉTRAITEMENT DES DONNÉES")
print("="*80)

# Supprimer les lignes avec des reviews vides ou des ratings manquants
df_clean = df_reviews.dropna(subset=['review', 'rating']).copy()
df_clean = df_clean[df_clean['review'].str.strip() != '']

print(f"\n✅ Après nettoyage: {len(df_clean):,} reviews ({len(df_clean)/len(df_reviews)*100:.1f}%)")

# Créer les labels de sentiment basés sur les ratings
# 1-2 étoiles = Négatif (0)
# 3 étoiles = Neutre (1)  
# 4-5 étoiles = Positif (2)
def rating_to_sentiment(rating):
    """Convertit une note (1-5) en label de sentiment (0-2)"""
    if rating <= 2:
        return 0  # Négatif
    elif rating == 3:
        return 1  # Neutre
    else:
        return 2  # Positif

df_clean['sentiment'] = df_clean['rating'].apply(rating_to_sentiment)
df_clean['sentiment_label'] = df_clean['sentiment'].map({
    0: 'Négatif',
    1: 'Neutre', 
    2: 'Positif'
})

# Afficher la distribution des sentiments
print(f"\n📊 Distribution des sentiments:")
sentiment_counts = df_clean['sentiment_label'].value_counts()
print(sentiment_counts)
print(f"\n   Positif: {sentiment_counts['Positif']/len(df_clean)*100:.1f}%")
print(f"   Neutre:  {sentiment_counts.get('Neutre', 0)/len(df_clean)*100:.1f}%")
print(f"   Négatif: {sentiment_counts.get('Négatif', 0)/len(df_clean)*100:.1f}%")

# Visualisation de la distribution
plt.figure(figsize=(10, 6))
sentiment_counts.plot(kind='bar', color=['#d62728', '#ff7f0e', '#2ca02c'])
plt.title('Distribution des Sentiments dans les Reviews', fontsize=14, fontweight='bold')
plt.xlabel('Sentiment', fontsize=12)
plt.ylabel('Nombre de Reviews', fontsize=12)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('sentiment_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

### =============================================================================
### 4. ÉCHANTILLONNAGE STRATIFIÉ (Pour accélérer l'entraînement)
### =============================================================================

In [ ]:
print("\n" + "="*80)
print("ÉCHANTILLONNAGE DES DONNÉES")
print("="*80)

# Pour le développement, on peut travailler avec un échantillon
# Augmentez ce nombre si vous avez plus de ressources GPU
SAMPLE_SIZE = 50000  # Ajustez selon vos besoins

# Échantillonnage stratifié pour conserver la distribution des sentiments
if len(df_clean) > SAMPLE_SIZE:
    df_sample = df_clean.groupby('sentiment', group_keys=False).apply(
        lambda x: x.sample(min(len(x), SAMPLE_SIZE // 3), random_state=42)
    ).reset_index(drop=True)
    print(f"\n📦 Échantillon créé: {len(df_sample):,} reviews")
else:
    df_sample = df_clean.copy()
    print(f"\n📦 Utilisation de toutes les données: {len(df_sample):,} reviews")

print(f"\n📊 Distribution dans l'échantillon:")
print(df_sample['sentiment_label'].value_counts())

### =============================================================================
### 5. SPLIT TRAIN/VALIDATION/TEST
### =============================================================================

In [ ]:
print("\n" + "="*80)
print("SPLIT DES DONNÉES")
print("="*80)

# Split 70% train, 15% validation, 15% test
train_df, temp_df = train_test_split(
    df_sample, 
    test_size=0.3, 
    random_state=42, 
    stratify=df_sample['sentiment']
)

val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5, 
    random_state=42, 
    stratify=temp_df['sentiment']
)

print(f"\n✅ Split effectué:")
print(f"   Train:      {len(train_df):,} reviews ({len(train_df)/len(df_sample)*100:.1f}%)")
print(f"   Validation: {len(val_df):,} reviews ({len(val_df)/len(df_sample)*100:.1f}%)")
print(f"   Test:       {len(test_df):,} reviews ({len(test_df)/len(df_sample)*100:.1f}%)")

### =============================================================================
### 6. CRÉATION DU DATASET PYTORCH
### =============================================================================

In [ ]:
print("\n" + "="*80)
print("PRÉPARATION DU TOKENIZER ET DES DATASETS")
print("="*80)

# Charger le tokenizer RoBERTa
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
print(f"\n📥 Chargement du tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SentimentDataset(Dataset):
    """Dataset PyTorch pour l'analyse de sentiments"""
    
    def __init__(self, reviews, sentiments, tokenizer, max_length=256):
        self.reviews = reviews
        self.sentiments = sentiments
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.reviews)
    
    def __getitem__(self, idx):
        review = str(self.reviews[idx])
        sentiment = self.sentiments[idx]
        
        # Tokenization
        encoding = self.tokenizer(
            review,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(sentiment, dtype=torch.long)
        }

# Créer les datasets
MAX_LENGTH = 256  # Longueur maximale des reviews tokenisées

train_dataset = SentimentDataset(
    train_df['review'].values,
    train_df['sentiment'].values,
    tokenizer,
    MAX_LENGTH
)

val_dataset = SentimentDataset(
    val_df['review'].values,
    val_df['sentiment'].values,
    tokenizer,
    MAX_LENGTH
)

test_dataset = SentimentDataset(
    test_df['review'].values,
    test_df['sentiment'].values,
    tokenizer,
    MAX_LENGTH
)

print(f"✅ Datasets créés:")
print(f"   Train:      {len(train_dataset):,} exemples")
print(f"   Validation: {len(val_dataset):,} exemples")
print(f"   Test:       {len(test_dataset):,} exemples")

### =============================================================================
### 7. CHARGEMENT ET CONFIGURATION DU MODÈLE
### =============================================================================

In [ ]:
print("\n" + "="*80)
print("CHARGEMENT DU MODÈLE ROBERTA")
print("="*80)

# Charger le modèle RoBERTa pré-entraîné pour la classification
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3  # 3 classes: Négatif, Neutre, Positif
)

# Déplacer le modèle sur GPU si disponible
model.to(device)

print(f"✅ Modèle chargé et déplacé sur {device}")
print(f"   Nombre de paramètres: {sum(p.numel() for p in model.parameters()):,}")

### =============================================================================
### 8. CONFIGURATION DE L'ENTRAÎNEMENT
### =============================================================================

In [ ]:
print("\n" + "="*80)
print("CONFIGURATION DE L'ENTRAÎNEMENT")
print("="*80)

# Définir les arguments d'entraînement
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=16 if torch.cuda.is_available() else 8,
    per_device_eval_batch_size=32 if torch.cuda.is_available() else 16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=torch.cuda.is_available(),  # Mixed precision si GPU disponible
    report_to="none"
)

# Fonction de calcul des métriques
def compute_metrics(pred):
    """Calcule les métriques d'évaluation"""
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc}

# Créer le Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("✅ Trainer configuré avec les paramètres suivants:")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch size (train): {training_args.per_device_train_batch_size}")
print(f"   Batch size (eval): {training_args.per_device_eval_batch_size}")
print(f"   Mixed precision (FP16): {training_args.fp16}")

### =============================================================================
### 9. ENTRAÎNEMENT DU MODÈLE
### =============================================================================

In [ ]:
print("\n" + "="*80)
print("ENTRAÎNEMENT DU MODÈLE")
print("="*80)

# Lancer l'entraînement
print("\n🚀 Début de l'entraînement...")
trainer.train()

print("\n✅ Entraînement terminé!")

### =============================================================================
### 10. ÉVALUATION SUR LE TEST SET
### =============================================================================

In [ ]:
print("\n" + "="*80)
print("ÉVALUATION SUR LE TEST SET")
print("="*80)

# Évaluer sur le test set
test_results = trainer.evaluate(test_dataset)
print(f"\n📊 Résultats sur le test set:")
print(f"   Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"   Loss: {test_results['eval_loss']:.4f}")

# Prédictions détaillées
predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(-1)
true_labels = predictions.label_ids

# Rapport de classification détaillé
print(f"\n📋 Rapport de classification complet:")
target_names = ['Négatif', 'Neutre', 'Positif']
print(classification_report(true_labels, preds, target_names=target_names))

# Matrice de confusion
cm = confusion_matrix(true_labels, preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names)
plt.title('Matrice de Confusion - Analyse de Sentiments', fontsize=14, fontweight='bold')
plt.ylabel('Vrai Label', fontsize=12)
plt.xlabel('Label Prédit', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

### =============================================================================
### 11. PRÉDICTION SUR TOUT LE DATASET
### =============================================================================

In [ ]:
print("\n" + "="*80)
print("PRÉDICTION SUR TOUT LE DATASET")
print("="*80)

def predict_sentiments_batch(texts, model, tokenizer, batch_size=32):
    """Prédit les sentiments pour un lot de textes"""
    model.eval()
    predictions = []
    
    # Créer des batches
    for i in tqdm(range(0, len(texts), batch_size), desc="Prédiction"):
        batch_texts = texts[i:i+batch_size]
        
        # Tokenization
        encodings = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors='pt'
        )
        
        # Déplacer sur le device
        encodings = {k: v.to(device) for k, v in encodings.items()}
        
        # Prédiction
        with torch.no_grad():
            outputs = model(**encodings)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)
            predictions.extend(preds.cpu().numpy())
    
    return predictions

print(f"\n🔮 Prédiction des sentiments pour tout le dataset ({len(df_clean):,} reviews)...")
all_predictions = predict_sentiments_batch(
    df_clean['review'].tolist(),
    model,
    tokenizer,
    batch_size=32 if torch.cuda.is_available() else 16
)

# Ajouter les prédictions au dataframe
df_clean['sentiment_predicted'] = all_predictions
df_clean['sentiment_label_predicted'] = df_clean['sentiment_predicted'].map({
    0: 'Négatif',
    1: 'Neutre',
    2: 'Positif'
})

print("\n✅ Prédictions terminées!")
print(f"\n📊 Distribution des sentiments prédits:")
print(df_clean['sentiment_label_predicted'].value_counts())

### =============================================================================
### 12. SAUVEGARDE DES RÉSULTATS
### =============================================================================

In [ ]:
print("\n" + "="*80)
print("SAUVEGARDE DES RÉSULTATS")
print("="*80)

# Sauvegarder le modèle fine-tuné
model.save_pretrained('./sentiment_model_roberta')
tokenizer.save_pretrained('./sentiment_model_roberta')
print("✅ Modèle sauvegardé dans './sentiment_model_roberta'")

# Sauvegarder le dataset avec les sentiments prédits
# Sélectionner les colonnes importantes
output_columns = [
    'user_id', 'recipe_id', 'date', 'rating', 'review',
    'sentiment_predicted', 'sentiment_label_predicted'
]

df_output = df_clean[output_columns].copy()
df_output.to_csv('reviews_with_sentiments.csv', index=False)
print(f"✅ Dataset enrichi sauvegardé: 'reviews_with_sentiments.csv' ({len(df_output):,} lignes)")

# Sauvegarder aussi une version allégée (sans le texte des reviews pour économiser de l'espace)
df_light = df_clean[['user_id', 'recipe_id', 'rating', 'sentiment_predicted']].copy()
df_light.to_csv('reviews_sentiments_light.csv', index=False)
print(f"✅ Version allégée sauvegardée: 'reviews_sentiments_light.csv' ({len(df_light):,} lignes)")

### =============================================================================
### 13. STATISTIQUES FINALES ET INSIGHTS
### =============================================================================

In [ ]:
print("\n" + "="*80)
print("STATISTIQUES FINALES ET INSIGHTS")
print("="*80)

# Comparaison entre sentiments basés sur ratings vs prédits
comparison = pd.DataFrame({
    'Basé sur Rating': df_clean['sentiment_label'].value_counts(),
    'Prédit par Modèle': df_clean['sentiment_label_predicted'].value_counts()
})
print(f"\n📊 Comparaison des distributions:")
print(comparison)

# Accord entre rating et prédiction
agreement = (df_clean['sentiment'] == df_clean['sentiment_predicted']).mean()
print(f"\n🎯 Accord entre sentiment (rating) et prédiction: {agreement*100:.2f}%")

# Exemples de reviews par sentiment
print(f"\n💬 Exemples de reviews par sentiment prédit:")
for sent_label in ['Positif', 'Neutre', 'Négatif']:
    examples = df_clean[df_clean['sentiment_label_predicted'] == sent_label]['review'].head(2)
    print(f"\n{sent_label}:")
    for i, ex in enumerate(examples, 1):
        print(f"  {i}. {ex[:150]}...")

print("\n" + "="*80)
print("✨ ANALYSE DE SENTIMENTS TERMINÉE AVEC SUCCÈS! ✨")
print("="*80)
print("\nFichiers générés:")
print("  1. sentiment_model_roberta/ - Modèle fine-tuné")
print("  2. reviews_with_sentiments.csv - Dataset complet avec sentiments")
print("  3. reviews_sentiments_light.csv - Version allégée")
print("  4. sentiment_distribution.png - Visualisation de la distribution")
print("  5. confusion_matrix.png - Matrice de confusion")
print("\n🎉 Vous pouvez maintenant utiliser ces données pour les objectifs suivants:")
print("  - Objectif 1: Système de Recommandation Personnalisée")
print("  - Objectif 2: Prédiction de Popularité")
print("  - Objectif 4: Segmentation des Utilisateurs")